# Generación de datos CPMP — V9 (pipeline PDF refinado)

Implementa el pipeline descrito en `Gen_datos.pdf` con los refinamientos discutidos:

- `S` (pilas) **fijo por dataset** (requerido por `DefaultMovesAdapter`).
- `T` (altura física) **variable por instancia** ∈ [3, H_pad], padding al `H_pad` global.
- `F` (fill rate) y `k` (grupos) sorteados por instancia.
- Estado base ordenado con distribución **round-robin** + shuffle de columnas (sin sesgo espacial).
- Caminata inversa con **anti-ciclo**, **pasos adaptativos por bucket** y **sesgo hacia bloqueos** en buckets duros.
- Cuotas por bucket aplicadas al **total del dataset** (sin descartar si encaja en otro bucket con cuota libre).
- Etiqueta `Y` exactamente como `DefaultMovesAdapter` (multilabel sobre primeros movimientos óptimos) — **coherente con el entrenamiento actual**.
- Verificación de la solución (`cost != inf`) + chequeo de 0% bloqueados en el estado base.
- Metadata por instancia (bucket, %bloqueados, T, k, F, gap, dificultad) guardada en el `.data`.

Uso:
1. Ejecutar las celdas de setup (clonar, compilar, paths).
2. Ejecutar la celda del pipeline (define todas las funciones).
3. Ejecutar la celda de generación con la config deseada (default: 250k, S=5, H_pad=10, adapter=5D).

## 1. Setup (Colab)

In [ ]:
# Si corres en Colab desde cero, descomenta para clonar y entrar al repo.
# !git clone https://github.com/felipe-astudillo-s/CPMP-Transformer.git
# %cd CPMP-Transformer/CPMP-Framework

In [ ]:
# Compilar solver C++ (forzamos la compilación para evitar problemas de binarios preexistentes o sin permisos).
import os
!g++ Codigo_C_solver/Greedy.cpp Codigo_C_solver/Layout.cpp Codigo_C_solver/Bsg.cpp Codigo_C_solver/main_cpmp.cpp -o Codigo_C_solver/frg -O3 -std=c++11
!chmod +x Codigo_C_solver/frg
print('Solver listo:', os.path.abspath('Codigo_C_solver/frg'))

In [ ]:
import sys, os
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.append(src_path)

## 2. Pipeline V9

Toda la lógica vive en esta celda — modular, sin tocar `src/generation/`. Se apoya en:
- `cpmp.layout.Layout` para representar bahías y aplicar el solver.
- `generation.data.greedy` (greedy puro C++, para gap) y `generation.data.get_best_moves` (greedy+lookahead → `Y`).
- Cualquier `LayoutDataAdapter` + `MovesDataAdapter` (modularidad como hoy).

In [ ]:
import math
import random
import time
import copy
import numpy as np
import h5py
from collections import deque
from concurrent.futures import ProcessPoolExecutor
from functools import partial

from cpmp.layout import Layout
from generation.data import greedy as cpp_greedy, get_best_moves
from settings import DATA_FOLDER

# -------- Definición de buckets --------
# (idx, nombre, %_min, %_max, cuota, pasos_min_x_N, pasos_max_x_N)
# Cuotas actualizadas: más peso en dificil+extremo para acercarse al benchmark CVS.
BUCKETS = [
    (0, 'trivial', 0.0,  15.0, 0.10,  1.0,  3.0),
    (1, 'facil',   15.0, 35.0, 0.15,  3.0,  6.0),
    (2, 'medio',   35.0, 55.0, 0.25,  6.0, 10.0),
    (3, 'dificil', 55.0, 75.0, 0.25, 10.0, 15.0),
    (4, 'extremo', 75.0,100.0, 0.25, 15.0, 25.0),
]
HARD_BUCKETS = {2, 3, 4}  # sesgo hacia bloqueos cuando se apunta a estos

def assign_bucket(blocked_pct):
    """Devuelve el índice del bucket al que pertenece el %bloqueados."""
    if blocked_pct <= 15.0: return 0
    if blocked_pct <= 35.0: return 1
    if blocked_pct <= 55.0: return 2
    if blocked_pct <= 75.0: return 3
    return 4

def count_blocked(stacks):
    """%-blocked según el algoritmo del PDF."""
    blocked = 0
    total = 0
    for stack in stacks:
        total += len(stack)
        max_above = 0
        for c in reversed(stack):
            if c < max_above:
                blocked += 1
            if c > max_above:
                max_above = c
    pct = (blocked / total) * 100.0 if total > 0 else 0.0
    return blocked, pct

# -------- Sortear parámetros --------
def sample_params(S, H_pad, rng, T_min=3, F_min=0.50, F_max=0.95):
    """Sortea (T, F, k, N) garantizando N >= k y al menos 1 slot vacío.
    F_max=0.95 para cubrir instancias densas (benchmark CVS tiene F~0.71-0.83)."""
    for _ in range(50):
        T = rng.randint(T_min, H_pad)
        F = rng.uniform(F_min, F_max)
        cap = S * T
        N = int(math.floor(cap * F))
        if N >= cap:
            N = cap - 1  # al menos 1 slot vacío
        if N < max(2, S):  # mínimo razonable
            continue
        k_min = max(2, math.ceil(N / 4))
        k_max = N
        if k_min > k_max:
            continue
        k = rng.randint(k_min, k_max)
        return T, F, k, N
    raise RuntimeError(f'No se logró sortear params válidos para S={S}, H_pad={H_pad}')

# -------- Estado objetivo (ordenado) --------
def build_ordered_layout(S, T, N, k, rng):
    """Construye una bahía ordenada y permuta columnas.
    Distribución round-robin: cada pila recibe contenedores de todos los grupos
    en orden descendente de prioridad (no-creciente bottom→top)."""
    base = N // k
    extra = N % k
    counts = [base + (1 if p <= extra else 0) for p in range(1, k + 1)]
    containers = []
    for p in range(k, 0, -1):
        containers.extend([p] * counts[p - 1])
    stacks = [[] for _ in range(S)]
    for i, c in enumerate(containers):
        stacks[i % S].append(c)
    for s in stacks:
        for j in range(1, len(s)):
            assert s[j] <= s[j - 1], f'Pila no ordenada: {s}'
    rng.shuffle(stacks)
    blocked, pct = count_blocked(stacks)
    assert blocked == 0, f'Estado base con {blocked} bloqueados (pct={pct})'
    return stacks

# -------- Caminata inversa con anti-ciclo reforzado --------
def inverse_walk(stacks_initial, T, n_steps, target_bucket, rng, p_bias=0.7, history_len=6):
    """Aplica n_steps movimientos válidos sobre la bahía ordenada.

    Anti-ciclo reforzado: ventana deslizante de `history_len` estados recientes.
    En cada paso se priorizan movimientos cuyo estado resultado NO esté en la
    ventana; sólo si todos los candidatos ciclarían se acepta el ciclo (fallback).

    Sesgo de bloqueo: si target_bucket es duro, con prob p_bias se elige
    primero entre movimientos que ponen un contenedor de mayor valor sobre uno
    menor (genera desorden rápido).
    """
    stacks = [s[:] for s in stacks_initial]
    S = len(stacks)

    # Ventana de estados recientes (tuplas de tuplas = hashable)
    recent = deque(maxlen=history_len)
    recent.append(tuple(tuple(s) for s in stacks))

    use_bias = target_bucket in HARD_BUCKETS
    moves_done = 0
    attempts = 0
    max_attempts = max(20, 8 * n_steps)

    while moves_done < n_steps and attempts < max_attempts:
        attempts += 1

        feasible = [
            (i, j)
            for i in range(S) if stacks[i]
            for j in range(S) if i != j and len(stacks[j]) < T
        ]
        if not feasible:
            break

        # Construir lista de candidatos con sesgo opcional
        if use_bias and rng.random() < p_bias:
            blocking = [
                (i, j) for i, j in feasible
                if stacks[j] and stacks[i][-1] > stacks[j][-1]
            ]
            base_candidates = list(blocking) if blocking else list(feasible)
        else:
            base_candidates = list(feasible)
        rng.shuffle(base_candidates)

        # Filtrar candidatos que no llevan a un estado reciente (max 8 checks)
        def preview_key(i, j):
            """Calcula la clave del estado resultante de mover i→j sin copiar listas."""
            top = stacks[i][-1]
            return tuple(
                tuple(stacks[k][:-1])        if k == i else
                tuple(stacks[k]) + (top,)   if k == j else
                tuple(stacks[k])
                for k in range(S)
            )

        chosen = base_candidates[0]  # fallback si todo cicla
        for cand in base_candidates[:8]:
            if preview_key(*cand) not in recent:
                chosen = cand
                break

        i, j = chosen
        stacks[j].append(stacks[i].pop())
        recent.append(tuple(tuple(s) for s in stacks))
        moves_done += 1

    return stacks

# -------- Selección de bucket objetivo --------
def pick_target_bucket(remaining, rng):
    """Pick weighted-random sobre cuotas restantes (>0)."""
    items = [(b, c) for b, c in remaining.items() if c > 0]
    if not items:
        return None
    weights = [c for _, c in items]
    return rng.choices([b for b, _ in items], weights=weights, k=1)[0]

def sample_n_steps(target_bucket, N, rng):
    _, _, _, _, _, lo_x, hi_x = BUCKETS[target_bucket]
    lo = max(1, int(lo_x * N))
    hi = max(lo, int(hi_x * N))
    return rng.randint(lo, hi)

# -------- Padding del tensor S (T_phys → H_pad) --------
def pad_S_matrix(S_matrix, H_pad, pad_val=-1.0):
    """S_matrix con shape (S_stacks, T_phys, K). Pad dim 1 hasta H_pad."""
    t_phys = S_matrix.shape[1]
    if t_phys == H_pad:
        return S_matrix
    if t_phys > H_pad:
        raise ValueError(f'T_phys={t_phys} > H_pad={H_pad}')
    pad_shape = list(S_matrix.shape)
    pad_shape[1] = H_pad - t_phys
    pad = np.full(pad_shape, pad_val, dtype=S_matrix.dtype)
    return np.concatenate([S_matrix, pad], axis=1)

# -------- Generación de un estado (Fase 1) --------
def generate_one_state(S_stacks, H_pad, rng, target_bucket, max_walk_retries=3):
    """Sortea params, construye bahía ordenada y aplica caminata inversa.

    Reintenta hasta max_walk_retries veces desde la misma base para intentar
    aterrizar en target_bucket. Devuelve el resultado del último intento
    (puede no ser target_bucket si ningún intento lo consiguió).
    """
    T, F, k, N = sample_params(S_stacks, H_pad, rng)
    base = build_ordered_layout(S_stacks, T, N, k, rng)
    last_result = None
    for _ in range(max_walk_retries):
        n_steps = sample_n_steps(target_bucket, N, rng)
        # inverse_walk copia base internamente → cada reintento parte del mismo estado
        stacks = inverse_walk(base, T, n_steps, target_bucket, rng)
        _, blocked_pct = count_blocked(stacks)
        actual_bucket = assign_bucket(blocked_pct)
        last_result = (stacks, T, k, F, N, blocked_pct, actual_bucket)
        if actual_bucket == target_bucket:
            break  # aterrizado en el bucket objetivo, no hace falta reintentar
    return last_result

# -------- Worker para Fase 2 (resolver con C++) --------
def _solve_worker(arg):
    """arg = (idx, stacks, T, max_steps, with_gap)"""
    idx, stacks, T, max_steps, with_gap = arg
    layout = Layout([s[:] for s in stacks], T)
    if layout.unsorted_stacks == 0:
        return idx, None, float('inf'), float('inf')
    try:
        best_moves, cost = get_best_moves(layout, T, max_steps)
    except Exception as e:
        print(f"Error en worker {idx}: {e}")
        return idx, None, float('inf'), float('inf')
    if cost == float('inf'):
        return idx, None, float('inf'), float('inf')
    cost_pure = float('inf')
    if with_gap:
        try:
            layout_pure = Layout([s[:] for s in stacks], T)
            cost_pure = cpp_greedy(layout_pure, T, max_steps)
        except Exception as e:
            print(f"Error en worker {idx}: {e}")
            cost_pure = float('inf')
    return idx, best_moves, cost, cost_pure

def classify_difficulty(cost_full, cost_pure):
    if cost_pure == float('inf') or cost_full == float('inf'):
        return float('inf'), 'dificil'
    if cost_full <= 0:
        return 0.0, 'facil'
    gap = (cost_pure - cost_full) / cost_full
    if gap <= 0.0:
        return gap, 'facil'
    if gap <= 0.20:
        return gap, 'medio'
    return gap, 'dificil'

# -------- Pipeline principal --------
def generate_dataset_v9(
    output_name,
    S_stacks,
    H_pad,
    total_instances,
    layout_adapter,
    moves_adapter,
    max_steps=100,
    seed=42,
    compute_difficulty=False,
    n_workers=None,
    progress_every=2000,
):
    """Genera un dataset siguiendo el pipeline V9 y lo guarda como h5py."""
    rng = random.Random(seed)

    quotas = {b[0]: int(round(total_instances * b[4])) for b in BUCKETS}
    diff = total_instances - sum(quotas.values())
    quotas[2] += diff  # absorber redondeo en bucket medio
    remaining = dict(quotas)
    print('Cuotas objetivo por bucket:', {BUCKETS[b][1]: quotas[b] for b in quotas})

    layout_cls = layout_adapter.__class__
    moves_cls = moves_adapter.__class__

    # ---- Fase 1: generar estados respetando cuotas ----
    print(f'\n[Fase 1] Generando {total_instances} estados...')
    t0 = time.time()
    raw = []
    attempts = 0
    while sum(remaining.values()) > 0:
        attempts += 1
        target = pick_target_bucket(remaining, rng)
        if target is None:
            break
        result = generate_one_state(S_stacks, H_pad, rng, target)
        if result is None:
            continue
        stacks, T, k, F, N, blocked_pct, actual_bucket = result
        if remaining.get(actual_bucket, 0) <= 0:
            placed = False
            for b_alt in (actual_bucket - 1, actual_bucket + 1):
                if 0 <= b_alt < len(BUCKETS) and remaining.get(b_alt, 0) > 0:
                    lo, hi = BUCKETS[b_alt][2], BUCKETS[b_alt][3]
                    if abs(blocked_pct - (lo + hi) / 2) < 12:
                        actual_bucket = b_alt
                        placed = True
                        break
            if not placed:
                continue
        raw.append((stacks, T, k, F, N, blocked_pct, actual_bucket))
        remaining[actual_bucket] -= 1
        if len(raw) % progress_every == 0:
            elapsed = time.time() - t0
            print(f'  · {len(raw):>7}/{total_instances} | attempts={attempts} | '
                  f'remaining={ {BUCKETS[b][1]: remaining[b] for b in remaining} } | '
                  f'{elapsed:.1f}s')
    print(f'[Fase 1] {len(raw)} estados generados en {time.time()-t0:.1f}s '
          f'(intentos totales: {attempts}, tasa de aceptación: {len(raw)/max(1,attempts):.2%})')

    # ---- Fase 2: resolver con C++ en paralelo ----
    print(f'\n[Fase 2] Resolviendo {len(raw)} instancias con greedy+lookahead'
          + (' (+gap)' if compute_difficulty else '') + '...')
    t0 = time.time()
    args = [(i, raw[i][0], raw[i][1], max_steps, compute_difficulty) for i in range(len(raw))]
    solved = [None] * len(raw)
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for n_done, res in enumerate(ex.map(_solve_worker, args, chunksize=64), 1):
            solved[res[0]] = res
            if n_done % progress_every == 0:
                print(f'  · {n_done}/{len(raw)} | {time.time()-t0:.1f}s')
    print(f'[Fase 2] resuelto en {time.time()-t0:.1f}s')

    # ---- Fase 3: convertir con adapter, padear y empaquetar ----
    print(f'\n[Fase 3] Convirtiendo con {layout_cls.__name__} + {moves_cls.__name__}...')
    t0 = time.time()
    costs = []
    meta_T, meta_k, meta_F, meta_N, meta_bucket, meta_blocked = [], [], [], [], [], []
    meta_gap, meta_diff = [], []
    n_dropped = 0
    for i, item in enumerate(raw):
        stacks, T, k_val, F_val, N_val, blocked_pct, actual_bucket = item
        sol = solved[i]
        if sol is None or sol[1] is None or sol[2] == float('inf'):
            n_dropped += 1
            continue
        _, best_moves, cost_full, cost_pure = sol
        layout = Layout([s[:] for s in stacks], T)
        layout_vec = layout_cls.layout_2_vec(layout, T)
        S_matrix = pad_S_matrix(np.asarray(layout_vec[0]), H_pad)
        layout_vec = (S_matrix,) + tuple(layout_vec[1:])
        layout_adapter.add(layout_vec)
        moves_vec = moves_cls.moves_2_vec(best_moves, S_stacks)
        moves_adapter.add(moves_vec)
        costs.append(int(cost_full) if cost_full != float('inf') else -1)
        meta_T.append(T); meta_k.append(k_val); meta_F.append(F_val)
        meta_N.append(N_val); meta_bucket.append(actual_bucket); meta_blocked.append(blocked_pct)
        if compute_difficulty:
            gap, diff_label = classify_difficulty(cost_full, cost_pure)
            meta_gap.append(gap if gap != float('inf') else -1.0)
            meta_diff.append({'facil': 0, 'medio': 1, 'dificil': 2}[diff_label])
    print(f'[Fase 3] empaquetado en {time.time()-t0:.1f}s | descartados={n_dropped}')

    # ---- Guardar h5py ----
    layout_data = layout_adapter.get()
    moves_data = moves_adapter.get()
    data = {**layout_data, **moves_data}
    output_path = DATA_FOLDER / f'{output_name}.data'
    with h5py.File(output_path, 'w') as f:
        keys_order = [k for k in data.keys() if k != 'C']
        f.attrs['key_order'] = list(keys_order)
        f.attrs['S_stacks'] = S_stacks
        f.attrs['H_pad'] = H_pad
        f.attrs['pipeline'] = 'V9'
        f.attrs['compute_difficulty'] = compute_difficulty
        for key in data:
            f.create_dataset(key, data=data[key])
        f.create_dataset('C', data=np.asarray(costs, dtype=np.int32))
        f.create_dataset('meta_T', data=np.asarray(meta_T, dtype=np.int32))
        f.create_dataset('meta_k', data=np.asarray(meta_k, dtype=np.int32))
        f.create_dataset('meta_F', data=np.asarray(meta_F, dtype=np.float32))
        f.create_dataset('meta_N', data=np.asarray(meta_N, dtype=np.int32))
        f.create_dataset('meta_bucket', data=np.asarray(meta_bucket, dtype=np.int32))
        f.create_dataset('meta_blocked_pct', data=np.asarray(meta_blocked, dtype=np.float32))
        if compute_difficulty:
            f.create_dataset('meta_gap', data=np.asarray(meta_gap, dtype=np.float32))
            f.create_dataset('meta_difficulty', data=np.asarray(meta_diff, dtype=np.int32))
    print(f'\nDatos guardados en: {output_path} (Tamaño {layout_adapter.count()})')
    if meta_bucket:
        unique, counts_b = np.unique(meta_bucket, return_counts=True)
        dist = {BUCKETS[u][1]: int(c) for u, c in zip(unique, counts_b)}
        print('Distribucion final por bucket:', dist)
    return output_path


## 3. Smoke test rápido — gen multi-S + verificación + forward de V9

Antes de lanzar 250k, conviene un smoke test de ~3 minutos que genera un dataset chico con varios `S`, verifica que `pad_batch_collate` (el del entrenamiento real) los mezcla bien, y corre un forward de V9 sobre un batch.

Ejecuta las 3 celdas siguientes en orden. Si las 3 pasan, todo está conectado y puedes pasar a la sección 4 con confianza.

In [ ]:
# --- Smoke test 1/3: generar 3 datasets pequeños con S distinto ---
from generation.adapters import EnrichedStackMatrix5DAdapter, DefaultMovesAdapter

SMOKE_CONFIGS = [
    # (S, H_pad, total) — totales chicos para que termine rápido
    (4, 10, 800),
    (5, 10, 1200),
    (6, 10, 800),
]
SMOKE_PATHS = []
for S_cfg, Hp_cfg, total_cfg in SMOKE_CONFIGS:
    name = f'SMOKE_S{S_cfg}_H{Hp_cfg}_{total_cfg}'
    SMOKE_PATHS.append(name)
    generate_dataset_v9(
        output_name=name,
        S_stacks=S_cfg,
        H_pad=Hp_cfg,
        total_instances=total_cfg,
        layout_adapter=EnrichedStackMatrix5DAdapter(),
        moves_adapter=DefaultMovesAdapter(),
        max_steps=80,
        seed=42 + S_cfg,
        progress_every=400,
    )
print('\n\n[SMOKE 1/3] ✅ 3 datasets generados:', SMOKE_PATHS)

In [ ]:
# --- Smoke test 2/3: cargar con ConcatDataset + pad_batch_collate ---
import torch
from torch.utils.data import ConcatDataset, DataLoader
from preprocessing.dataset import H5Dataset
from training.training import pad_batch_collate
from settings import DATA_FOLDER

datasets = [H5Dataset(str(DATA_FOLDER / f'{name}.data')) for name in SMOKE_PATHS]
combined = ConcatDataset(datasets)
print(f'Total instancias combinadas: {len(combined)}')
for d in datasets:
    print(f'  · {d.name}: {len(d)} instancias')

loader = DataLoader(combined, batch_size=32, shuffle=True, collate_fn=pad_batch_collate)
batch = next(iter(loader))
S_b, X_b, Y_b = batch
print(f'\nBatch shapes:')
print(f'  S: {tuple(S_b.shape)}  (batch, max_S, max_H, C_dim)')
print(f'  X: {tuple(X_b.shape)}  (batch, max_S, X_dim)')
print(f'  Y: {tuple(Y_b.shape)}  (batch, max_S*(max_S-1))')
assert (S_b == -1).any().item(), 'S debe contener -1 si hay padding'
assert S_b.shape[1] == X_b.shape[1], 'max_S debe coincidir entre S y X'
assert Y_b.shape[1] == S_b.shape[1] * (S_b.shape[1] - 1), 'Y debe tener max_S*(max_S-1)'
print('\n[SMOKE 2/3] ✅ pad_batch_collate mezcla S=4/5/6 correctamente')

In [ ]:
# --- Smoke test 3/3: forward V9 sobre el batch mixto ---
from models.cpmp_transformer_v9 import CPMPTransformer

model_v9 = CPMPTransformer(H=10, C_dim=2, X_dim=5, d_model=64, nhead=8, num_layers=2)
model_v9.eval()
with torch.no_grad():
    logits = model_v9(S_b.float(), X_b.float())
expected_actions = S_b.shape[1] * (S_b.shape[1] - 1)
print(f'Logits shape: {tuple(logits.shape)}  (esperado: ({S_b.shape[0]}, {expected_actions}))')
assert logits.shape == (S_b.shape[0], expected_actions), 'Shape de logits incorrecto'
assert not torch.isnan(logits).any().item(), 'Logits con NaN'
assert not torch.isinf(logits).any().item(), 'Logits con Inf'
print('\n[SMOKE 3/3] ✅ V9 forward OK con batch de S mixto')
print('\n🎉 Smoke test completo — todo conectado. Puedes pasar a la generación de 250k.')

## 4. Generación principal — 250k instancias


In [ ]:
from generation.adapters import (
    EnrichedStackMatrix5DAdapter,
    DefaultMovesAdapter,
)

# Ejemplo de generación para un S fijo. Para generar el dataset completo
# con múltiples S (incluyendo S=3,9,10) ver la sección 6 (Multi-config).
OUTPUT_NAME      = 'V9_S5_H12_80k_5D'
S_STACKS         = 5
H_PAD            = 12
TOTAL_INSTANCES  = 80_000
MAX_STEPS        = 100
SEED             = 42
COMPUTE_DIFF     = False

layout_adapter = EnrichedStackMatrix5DAdapter()
moves_adapter  = DefaultMovesAdapter()

generate_dataset_v9(
    output_name=OUTPUT_NAME,
    S_stacks=S_STACKS,
    H_pad=H_PAD,
    total_instances=TOTAL_INSTANCES,
    layout_adapter=layout_adapter,
    moves_adapter=moves_adapter,
    max_steps=MAX_STEPS,
    seed=SEED,
    compute_difficulty=COMPUTE_DIFF,
)

## 5. Regenerar el mismo dataset con otro adaptador (modularidad)


In [ ]:
# Ejemplo: mismo dataset con adapter táctico (X de 5 dims tácticas en lugar de las 5 del 5D enriched)
# generate_dataset_v9(
#     output_name='V9_S5_H10_250k_TACTICAL',
#     S_stacks=S_STACKS,
#     H_pad=H_PAD,
#     total_instances=TOTAL_INSTANCES,
#     layout_adapter=TacticalStackMatrixAdapter(),
#     moves_adapter=DefaultMovesAdapter(),
#     max_steps=MAX_STEPS,
#     seed=SEED,  # mismo seed → mismas instancias
# )

## 6. Multi-config (varios `S`)


In [ ]:
from generation.adapters import EnrichedStackMatrix5DAdapter, DefaultMovesAdapter

# Dataset completo cubriendo el rango de S del benchmark CVS (3..10).
# Total: ~380k instancias. Ajusta los totales según memoria/tiempo disponibles.
configs = [
    # (S, H_pad, total, seed)
    ( 3, 12,  20_000, 45),   # S=3: instancias pequeñas (benchmark 3-3..3-8)
    ( 4, 12,  50_000, 46),   # S=4
    ( 5, 12,  80_000, 47),   # S=5 (mayor peso: más categorías CVS)
    ( 6, 12,  80_000, 48),   # S=6
    ( 7, 12,  60_000, 49),   # S=7
    ( 8, 12,  40_000, 50),   # S=8
    ( 9, 12,  30_000, 51),   # S=9: benchmark 5-9
    (10, 12,  20_000, 52),   # S=10: benchmark 5-10, 6-10, 10-10
]

for S_cfg, Hp_cfg, total_cfg, seed_cfg in configs:
    name = f'V9_DEFINITIVO_S{S_cfg}_H{Hp_cfg}_{total_cfg//1000}k'
    print(f'\n{"="*60}\nGenerando {name} ...\n{"="*60}')
    generate_dataset_v9(
        output_name=name,
        S_stacks=S_cfg,
        H_pad=Hp_cfg,
        total_instances=total_cfg,
        layout_adapter=EnrichedStackMatrix5DAdapter(),
        moves_adapter=DefaultMovesAdapter(),
        max_steps=100,
        seed=seed_cfg,
    )
print('\nTodos los datasets generados.')

## 7. Inspección rápida del `.data` generado

In [ ]:
# import h5py
# with h5py.File(DATA_FOLDER / f'{OUTPUT_NAME}.data', 'r') as f:
#     print('Keys:', list(f.keys()))
#     print('Attrs:', dict(f.attrs))
#     for k in f.keys():
#         print(f'  {k}: shape={f[k].shape}, dtype={f[k].dtype}')
#     bucket = f['meta_bucket'][:]
#     blocked = f['meta_blocked_pct'][:]
#     T = f['meta_T'][:]
#     unique, counts = np.unique(bucket, return_counts=True)
#     print('\nDistribución por bucket:')
#     for u, c in zip(unique, counts):
#         print(f'  {BUCKETS[u][1]:8s}: {c:>7} ({c/len(bucket):.1%})')
#     print(f'\n%bloqueados: media={blocked.mean():.1f} mediana={np.median(blocked):.1f} std={blocked.std():.1f}')
#     print(f'T (altura física): media={T.mean():.2f} min={T.min()} max={T.max()}')